In [ ]:
# Enumerator
# the purpose is to store the data in the circular buffer and yield 


%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append(os.environ['HOME'] + "/repos/dataengine/")
    
from source import DataSources
from buffer import DataBuffer
from iterator import DataIterator

# from rosbags.rosbag2 import Reader
from rosbags.highlevel import AnyReader
from rosbags.serde import deserialize_cdr

import numpy as np
from pathlib import Path


## TODO:

# 1. create data sources class/object
# 2. create buffer class/object
# 3. create enumeration class/object
# 4. create image registration application step
# 5. then the rest of the steps from "below horizon"?



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:

## Source class
# Enumerate over png files and yield pointer pointer to circular buffer


## PNG files
# file_path = "/data/hpc/amazon/844f3538f69041158a5054eee612b75c/*.png"

# ## ROS1
# file_path = "/data/lbnl/roberts/data.bag"
# file_path = "/data/lbnl/2021_04_21_15_26_16/data.bag"
# file_path = "/data/lbnl/polaris_cher_indoors/data.bag"

# ## ROS2
# file_path = "/data/oak/"
# file_path = "/data/lbnl/roberts/output/data.db3"

# ## ROS2 LOAM
file_path = "/data/LOAM_ROS_2/2018-05-18-14-49-12_0/2018-05-18-14-49-12_0.db3"

data = DataSources(file_path)


In [5]:

## Buffer class


# depth of buffer as message count 
buffer_depth = 50
buffer = DataBuffer(data, buffer_depth)

#####

# print("topics: ", buffer.get_topics())

# axis = "/velodyne/velodyne_points"
# buffer.roll_buffer(axis)

# T = 1
# while T > 0:
#     last_message = buffer[0]
#     print(last_message.dtype, last_message.shape)
#     buf = buffer.get_buffer()
#     for key in buf:
#         print("key: ", key, " -- data: ", buf[key])
#     T -= 1


In [6]:

## Enumeration class

axis = "/velodyne_points"

iterator = DataIterator(buffer, axis)


In [7]:

## ICP example

for buf in iterator.get_data():
    # print(list(buf.keys()))
    # print(buf[list(buf.keys())[0]])
    pass


End of source


In [8]:

## LOAM example

for buf in iterator.get_data():
    # print(list(buf.keys()))
    # print(buf[list(buf.keys())[0]])
    pass


End of source


In [9]:

from __future__ import annotations

import importlib
from typing import TYPE_CHECKING

import numpy

if TYPE_CHECKING:
    from typing import Any

NATIVE_CLASSES: dict[str, Any] = {}


def to_native(msg: Any) -> Any:  # noqa: ANN401
    """Convert rosbags message to native message.

    Args:
        msg: Rosbags message.

    Returns:
        Native message.

    """
    msgtype: str = msg.__msgtype__
    if msgtype not in NATIVE_CLASSES:
        pkg, name = msgtype.rsplit('/', 1)
        NATIVE_CLASSES[msgtype] = getattr(importlib.import_module(pkg.replace('/', '.')), name)

    fields = {}
    for name, field in msg.__dataclass_fields__.items():
        if 'ClassVar' in field.type:
            continue
        value = getattr(msg, name)
        if '__msg__' in field.type:
            value = to_native(value)
        elif isinstance(value, list):
            value = [to_native(x) for x in value]
        elif isinstance(value, numpy.ndarray):
            value = value.tolist()
        fields[name] = value

    return NATIVE_CLASSES[msgtype](**fields)


In [11]:

from sensor_msgs.msg import PointCloud2, PointField

from sensor_msgs.msg import Image
import ros2_numpy as rnp

allowed_msgs = ["PointCloud2", "Image", "OccupancyGrid", "Vector3", "Point", "Quaternion", "Transform", "Pose"]

with AnyReader([Path(os.environ['HOME'] + '/repos/monorepo/external_files/data/oak/')]) as reader:
    for connection, timestamp, rawdata in reader.messages():
        # if connection.topic == '/oak1_left/image_raw':
        msg = deserialize_cdr(rawdata, connection.msgtype)
        native_image = to_native(msg)

        if msg.__class__.__name__ in allowed_msgs:
            npified = rnp.numpify(native_image)
    
            print(npified.shape)

            break

        
        # np_arr = np.frombuffer(msg.data, name_to_dtypes[msg.encoding])

        # print(dir(msg))

        # print("msg class: ", msg.__class__.__name__)
        # print("image class: ", Image.__class__.__name__)

        # print(np_arr.shape)
        # print(np_arr[0,0])
        



In [ ]:


# Enumerate over TileDB object and yield pointer to circular buffer

# with tiledb.open(dense_uri, mode="r") as arr:
#     val = arr[0:10, 0:10]
    


In [105]:

# type_mappings = [(PointField.INT8, np.dtype('int8')),
#                  (PointField.UINT8, np.dtype('uint8')),
#                  (PointField.INT16, np.dtype('int16')),
#                  (PointField.UINT16, np.dtype('uint16')),
#                  (PointField.INT32, np.dtype('int32')),
#                  (PointField.UINT32, np.dtype('uint32')),
#                  (PointField.FLOAT32, np.dtype('float32')),
#                  (PointField.FLOAT64, np.dtype('float64'))]
# pftype_to_nptype = dict(type_mappings)
# nptype_to_pftype = dict((nptype, pftype) for pftype, nptype in type_mappings)

# print(type_mappings)
# print()
# print(pftype_to_nptype)
# print()
# print(nptype_to_pftype)
# print()



# import ros2_numpy as rnp
# npified = rnp.numpify(msg)



In [ ]:

# from sensor_msgs.msg import PointCloud2, PointField
# import sensor_msgs.point_cloud2 as pc2
# import ctypes
# import struct
# from sensor_msgs.msg import PointCloud2, PointField
# from std_msgs.msg import Header


#       def __init__(self):
#             '''initiliaze  ros stuff '''

#             self.cloud_sub = rospy.Subscriber("/your_cloud_topic", PointCloud2,self.callback,queue_size=1, buff_size=52428800)      

#         def callback(self, ros_point_cloud):
#                 xyz = np.array([[0,0,0]])
#                 rgb = np.array([[0,0,0]])
#                 #self.lock.acquire()
#                 gen = pc2.read_points(ros_point_cloud, skip_nans=True)
#                 int_data = list(gen)

#                 for x in int_data:
#                     test = x[3] 
#                     # cast float32 to int so that bitwise operations are possible
#                     s = struct.pack('>f' ,test)
#                     i = struct.unpack('>l',s)[0]
#                     # you can get back the float value by the inverse operations
#                     pack = ctypes.c_uint32(i).value
#                     r = (pack & 0x00FF0000)>> 16
#                     g = (pack & 0x0000FF00)>> 8
#                     b = (pack & 0x000000FF)
#                     # prints r,g,b values in the 0-255 range
#                                 # x,y,z can be retrieved from the x[0],x[1],x[2]
#                     xyz = np.append(xyz,[[x[0],x[1],x[2]]], axis = 0)
#                     rgb = np.append(rgb,[[r,g,b]], axis = 0)

In [10]:


# name_to_dtypes = {
#     "rgb8":    (np.uint8,  3),
#     "rgba8":   (np.uint8,  4),
#     "rgb16":   (np.uint16, 3),
#     "rgba16":  (np.uint16, 4),
#     "bgr8":    (np.uint8,  3),
#     "bgra8":   (np.uint8,  4),
#     "bgr16":   (np.uint16, 3),
#     "bgra16":  (np.uint16, 4),
#     "mono8":   (np.uint8,  1),
#     "mono16":  (np.uint16, 1),
    
# }


In [11]:
# axis = "/camera/image_raw"
# update = buffer.roll_buffer(axis)
# cam_data = update[axis]

# print("keys: ", list(update.keys()))
# print("cam data: ", cam_data.shape, cam_data.dtype)
# print("vel data: ", vel_data.shape, vel_data.dtype)

# print("last array on axis: ", buffer[-1].shape)
# print("last second on axis: ", buffer[1.0].shape)

